# 02_Preprocessing.ipynb
## FakeJobShield: Text Cleaning and Preprocessing Pipeline
This notebook implements a complete text preprocessing pipeline, including: lowercasing, HTML tags removal, URL/email removal, special character cleaning, stopword filtering, tokenization, and lemmatization. It also combines multiple text fields to form a single combined textual feature.


In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import os

# Adjust working directory if run from notebooks folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)


In [ ]:
# Load dataset
df = pd.read_csv("data/fake_job_postings.csv")
print("Initial dataset shape:", df.shape)


In [ ]:
# Define combined text field
text_cols = ["title", "company_profile", "description", "requirements", "benefits"]

for col in text_cols:
    df[col] = df[col].fillna("").astype(str)

# Combine fields with spaces
df["combined_text"] = df["title"] + " " + df["company_profile"] + " " + df["description"] + " " + df["requirements"] + " " + df["benefits"]
print("Combined text sample (first 100 chars):")
print(df["combined_text"].iloc[0][:200])


In [ ]:
# Custom preprocessing function
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove HTML Tags
    text = re.sub(r"<[^>]*>", " ", text)
    
    # 3. Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    
    # 4. Remove Emails
    text = re.sub(r"\S+@\S+", " ", text)
    
    # 5. Remove Special Characters & Digits, keep letters and spaces
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    
    # 6. Tokenize & Stopwords & Lemmatize
    tokens = nltk.word_tokenize(text)
    cleaned_tokens = [
        lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2
    ]
    
    return " ".join(cleaned_tokens)


In [ ]:
# Test cleaning function
sample = "Check out our site at https://example.com/careers or contact recruiters@example.com for <b>Data Scientist</b> positions!"
print("Original:\n", sample)
print("Cleaned:\n", clean_text(sample))


In [ ]:
# Apply preprocessing
print("Preprocessing combined_text...")
df["cleaned_text"] = df["combined_text"].apply(clean_text)
print("Preprocessing complete!")


In [ ]:
# Preview cleaned text vs original
print("Original Combined Text:")
print(df["combined_text"].iloc[1][:150])
print("\nCleaned Combined Text:")
print(df["cleaned_text"].iloc[1][:150])


In [ ]:
# Save the cleaned dataset
os.makedirs("data", exist_ok=True)
df.to_csv("data/cleaned_fake_job_postings.csv", index=False)
print("Cleaned dataset saved successfully to 'data/cleaned_fake_job_postings.csv'")
